In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from scipy.sparse import csr_matrix
from sklearn.metrics import ndcg_score


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "trained_rl_model"

DATA_FILE = BASE_DIR / "test_reference.csv"
PREPROCESSOR_FILE = BASE_DIR / "preprocessor.joblib"

OUTPUT_FILE = BASE_DIR / "top4_recommendations.csv"
METRICS_FILE = BASE_DIR / "recommendation_metrics.json"

BATCH_SIZE = 2048
TOP_K = 4

numeric_features = [
    "price",
    "area",
    "price_per_sqft",
    "bedroom_num",
    "bathroom_num",
    "balcony_num",
    "age",
    "budget_max",
    "min_area",
    "max_area",
    "bedroom_pref",
    "bathroom_pref",
    "balcony_pref",
    "max_age",
]

categorical_features = [
    "locality",
    "city",
    "property_type",
    "furnished",
    "locality_pref",
    "city_pref",
    "property_type_pref",
    "furnished_pref",
]

feature_columns = numeric_features + categorical_features

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("REAL ESTATE TOP 4 RECOMMENDATION SYSTEM")
print("=" * 65)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

required_files = [
    DATA_FILE,
    PREPROCESSOR_FILE,
    MODEL_DIR / "best_model.pth",
    MODEL_DIR / "model_config.json",
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Check the file location before running again."
        )

with open(MODEL_DIR / "model_config.json", "r") as f:
    config = json.load(f)

preprocessor = joblib.load(PREPROCESSOR_FILE)

print("\nModel configuration loaded.")
print("Expected input features:", config["input_features"])


# ============================================================
# 3. RECREATE THE TRAINED NEURAL NETWORK
# ============================================================

class PropertyValueNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


model = PropertyValueNetwork(
    config["input_features"]
).to(device)

try:
    checkpoint = torch.load(
        MODEL_DIR / "best_model.pth",
        map_location=device,
        weights_only=True,
    )
except TypeError:
    # Compatibility with older PyTorch versions.
    checkpoint = torch.load(
        MODEL_DIR / "best_model.pth",
        map_location=device,
    )

model.load_state_dict(checkpoint)
model.eval()

print("Trained model loaded successfully.")


# ============================================================
# 4. LOAD TEST USERS AND THEIR CANDIDATE PROPERTIES
# ============================================================

df = pd.read_csv(DATA_FILE)

required_columns = [
    "user_id",
    "property_id",
    "title",
    "reward",
] + feature_columns

missing = sorted(set(required_columns) - set(df.columns))

if missing:
    raise ValueError(
        f"Missing columns in test_reference.csv: {missing}"
    )

if df[required_columns].isnull().any().any():
    raise ValueError(
        "Missing values detected in required columns. "
        "Inspect the reference CSV before proceeding."
    )

print("\nTest rows loaded :", f"{len(df):,}")
print("Unique test users:", f"{df['user_id'].nunique():,}")


# Apply exactly the same text standardisation used during
# preprocessing. Do not refit the preprocessor.
for col in categorical_features:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("", "unknown")
    )

X = df[feature_columns]

# Reuse the fitted training preprocessor.
X_processed = csr_matrix(
    preprocessor.transform(X),
    dtype=np.float32,
)

if X_processed.shape[1] != config["input_features"]:
    raise ValueError(
        f"Expected {config['input_features']} features, "
        f"but received {X_processed.shape[1]}."
    )

print("Processed feature matrix:", X_processed.shape)


# ============================================================
# 5. PREDICT REWARD SCORES IN BATCHES
# ============================================================

predictions = []

with torch.no_grad():
    for start in range(0, X_processed.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, X_processed.shape[0])

        batch = X_processed[start:end].toarray().astype(
            np.float32, copy=False
        )

        batch_tensor = torch.from_numpy(batch).to(device)

        batch_scores = model(batch_tensor)

        predictions.extend(
            batch_scores.cpu().numpy().tolist()
        )

df["predicted_reward"] = np.asarray(
    predictions, dtype=np.float32
)

print("Reward prediction complete.")


# ============================================================
# 6. RANK PROPERTIES AND SELECT TOP 4 PER USER
# ============================================================

# Higher predicted reward = stronger predicted match.
ranked = df.sort_values(
    ["user_id", "predicted_reward", "property_id"],
    ascending=[True, False, True],
).copy()

ranked["recommendation_rank"] = (
    ranked.groupby("user_id").cumcount() + 1
)

top4 = ranked[
    ranked["recommendation_rank"] <= TOP_K
].copy()

# Select readable, useful columns for the output.
output_columns = [
    "user_id",
    "recommendation_rank",
    "property_id",
    "title",
    "price",
    "area",
    "bedroom_num",
    "bathroom_num",
    "locality",
    "city",
    "property_type",
    "furnished",
    "predicted_reward",
    "reward",
]

top4[output_columns].to_csv(
    OUTPUT_FILE,
    index=False,
)

print("\nTop 4 recommendations saved to:")
print(OUTPUT_FILE)


# ============================================================
# 7. EVALUATE RANKING QUALITY WITH NDCG@4
# ============================================================

# The original simulated reward acts as the relevance label.
# We compare the model's ranking with the reward-based ranking.
ndcg_values = []
random_ndcg_values = []

rng = np.random.default_rng(42)

for user_id, group in ranked.groupby("user_id"):
    if len(group) < TOP_K:
        continue

    true_rewards = group["reward"].to_numpy(dtype=float)
    predicted_scores = group[
        "predicted_reward"
    ].to_numpy(dtype=float)

    ndcg = ndcg_score(
        [true_rewards],
        [predicted_scores],
        k=TOP_K,
    )

    ndcg_values.append(float(ndcg))

    # Random-order baseline for the same candidate pool.
    random_scores = rng.random(len(group))

    random_ndcg = ndcg_score(
        [true_rewards],
        [random_scores],
        k=TOP_K,
    )

    random_ndcg_values.append(float(random_ndcg))

mean_ndcg = float(np.mean(ndcg_values))
mean_random_ndcg = float(np.mean(random_ndcg_values))

metrics = {
    "evaluation_dataset": DATA_FILE.name,
    "test_users_evaluated": len(ndcg_values),
    "candidates_per_user": int(
        ranked.groupby("user_id").size().median()
    ),
    "top_k": TOP_K,
    "mean_ndcg_at_4": mean_ndcg,
    "random_baseline_ndcg_at_4": mean_random_ndcg,
    "interpretation": (
        "NDCG@4 measures how well the model ranks high-reward "
        "properties near the top. The random baseline is a "
        "comparison, not a trained recommendation model."
    ),
}

with open(METRICS_FILE, "w") as f:
    json.dump(metrics, f, indent=4)

print("\n" + "=" * 65)
print("RECOMMENDATION EVALUATION")
print("=" * 65)
print(f"Users evaluated       : {len(ndcg_values):,}")
print(f"Mean NDCG@4            : {mean_ndcg:.4f}")
print(f"Random baseline NDCG@4 : {mean_random_ndcg:.4f}")

print("\nMetrics saved to:")
print(METRICS_FILE)


# ============================================================
# 8. DISPLAY EXAMPLE RECOMMENDATIONS
# ============================================================

print("\n" + "=" * 65)
print("SAMPLE TOP 4 RECOMMENDATIONS")
print("=" * 65)

for user_id in top4["user_id"].drop_duplicates().head(3):
    group = top4[top4["user_id"] == user_id]

    print(f"\nUSER: {user_id}")

    for _, row in group.iterrows():
        print(
            f"\nRank #{int(row['recommendation_rank'])}: "
            f"{row['title']}"
        )
        print(f"  Price             : Rs. {row['price']:,.0f}")
        print(f"  Area              : {row['area']:,.0f} sq.ft")
        print(f"  BHK               : {row['bedroom_num']}")
        print(f"  Locality          : {row['locality']}")
        print(
            f"  Predicted reward  : "
            f"{row['predicted_reward']:.4f}"
        )

print("\n" + "=" * 65)
print("TOP 4 RECOMMENDATION RUN COMPLETE")
print("=" * 65)
print("Created:")
print(" -", OUTPUT_FILE.name)
print(" -", METRICS_FILE.name)

REAL ESTATE TOP 4 RECOMMENDATION SYSTEM
Device: cuda
GPU: NVIDIA H100 PCIe MIG 3g.40gb

Model configuration loaded.
Expected input features: 497
Trained model loaded successfully.


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.7.1 when using version 1.7.2. This might lead to breaking code o


Test rows loaded : 50,000
Unique test users: 1,000
Processed feature matrix: (50000, 497)
Reward prediction complete.

Top 4 recommendations saved to:
/home/mpstme-kanishka/pbs.179984.srv-svkmmastermum.x8z/top4_recommendations.csv

RECOMMENDATION EVALUATION
Users evaluated       : 1,000
Mean NDCG@4            : 0.9994
Random baseline NDCG@4 : 0.7080

Metrics saved to:
/home/mpstme-kanishka/pbs.179984.srv-svkmmastermum.x8z/recommendation_metrics.json

SAMPLE TOP 4 RECOMMENDATIONS

USER: U000001

Rank #1: Techno Vision Vivanta Marvel
  Price             : Rs. 13,800,000
  Area              : 1,210 sq.ft
  BHK               : 4.0
  Locality          : mira road
  Predicted reward  : 0.9393

Rank #2: Aristone Vasudev Paradise
  Price             : Rs. 14,600,000
  Area              : 1,400 sq.ft
  BHK               : 3.0
  Locality          : mira road
  Predicted reward  : 0.9238

Rank #3: SK Imperial Garden
  Price             : Rs. 11,680,000
  Area              : 1,412 sq.ft
  BHK    